# Classroom Compass — 04 · Report Builder

Builds the self-contained HTML insight report from a completed pipeline run.

**Prerequisites:** `report_template.html` and `chart.umd.min.js` in the project root.

**Sections**
1. Data Loading — run outputs + supplemental tables
2. User-Defined Functions — all UDFs consolidated here
3. Chart Data — per-insight demographic / resource distributions
4. Distinctness (TVD) — how distinct each insight is vs full population
5. Map Export — per-insight state distribution CSV
6. Report Builder — assemble JSON payload and render HTML

In [1]:
import sys, json, math, base64
from pathlib import Path

import pandas as pd
import numpy as np

ROOT = Path(".")
sys.path.insert(0, str(ROOT))

# ── Load versioned utils module ───────────────────────────────────────────
import importlib.util as _ilu

_utils_path = next(Path(".").glob("utils*.py"))
if _utils_path.stem != "utils":
    _spec = _ilu.spec_from_file_location("utils", _utils_path)
    _mod  = _ilu.module_from_spec(_spec)
    _spec.loader.exec_module(_mod)
    sys.modules["utils"] = _mod

from utils import load_cfg, resolve_params_path

CFG_PATH = resolve_params_path()
CFG      = load_cfg(CFG_PATH)

## 1 · Data Loading

In [2]:
# ── Set this to the run output directory you want to report on ─────────────
RUN_OUTPUT_ROOT = Path("OUTPUTS/runs/project_category/2026-04-11/20260411_201415_project_category_77e319db")

# Run identity from pipeline manifest
_manifest         = json.loads((RUN_OUTPUT_ROOT / "metadata" / "pipeline_manifest.json").read_text())
RUN_ID            = _manifest["run_id"]
RUN_DATE          = _manifest["run_date"]
GROUPBY_FIELD     = _manifest["group_by_field"]
FILTER_FIELDS_KEY = _manifest["filter_fields_key"]

def OUT(subdir, fname):
    """Return path under RUN_OUTPUT_ROOT/subdir, creating it if needed."""
    p = RUN_OUTPUT_ROOT / subdir
    p.mkdir(parents=True, exist_ok=True)
    return p / fname

# ── Core pipeline outputs ─────────────────────────────────────────────────
df                 = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")
insight_project_df = pd.read_csv(OUT("insights", "insight_project_bridge.csv"))
curated_df         = pd.read_csv(OUT("chart_data", "curated_df.csv"))
structured         = json.loads(OUT("insights", "insights_structured.json").read_text())
insights_flat_df   = pd.read_csv(OUT("insights", "insights_flat.csv"))

# ── Supplemental data ─────────────────────────────────────────────────────
resource_category_df  = pd.read_csv(ROOT / "DATA/project_resource_categorylevel.csv")
resource_item_df      = pd.read_csv(ROOT / "DATA/project_resource_itemlevel.csv")
project_attributes_df = pd.read_csv(ROOT / "DATA/project_attributes.csv",
                                     usecols=["project_id", "state"])

# Attach 2-letter state abbreviations to the main project table
df = df.merge(project_attributes_df, on="project_id", how="left")

print(f"RUN_ID             = {RUN_ID}")
print(f"GROUPBY_FIELD      = {GROUPBY_FIELD!r}")
print(f"df                 = {len(df):,} rows")
print(f"insight_project_df = {len(insight_project_df):,} rows")
print(f"curated_df         = {len(curated_df):,} rows")
print(f"insights_flat_df   = {len(insights_flat_df):,} rows")
print(f"resource_category  = {len(resource_category_df):,} rows")
print(f"resource_item      = {len(resource_item_df):,} rows")
print(f"state nulls        = {df['state'].isna().sum():,}")

# Analysis scope: the filtered project count written by notebook 03.
# filter_summary.json records output_row_count = rows after applying params.yaml filters
# (e.g. funded_date >= 2023-07-01), which is the 420k figure shown in NB03's output.
_fsum_path = RUN_OUTPUT_ROOT / "metadata" / "filter_summary.json"
if _fsum_path.exists():
    _fsum     = json.loads(_fsum_path.read_text())
    ANALYSIS_N = int(_fsum.get("output_row_count", insight_project_df["project_id"].nunique()))
else:
    # Fallback: distinct projects appearing in any insight
    ANALYSIS_N = insight_project_df["project_id"].nunique()
    print("Warning: filter_summary.json not found — using insight bridge count as fallback")
print(f"ANALYSIS_N         = {ANALYSIS_N:,}  (source: filter_summary.json)")

RUN_ID             = 20260411_201415_project_category_77e319db
GROUPBY_FIELD      = 'project_category'
df                 = 2,255,909 rows
insight_project_df = 159,315 rows
curated_df         = 47 rows
insights_flat_df   = 50 rows
resource_category  = 15,227,814 rows
resource_item      = 23,081,390 rows
state nulls        = 1,198
ANALYSIS_N         = 420,625  (source: filter_summary.json)


## 2 · User-Defined Functions

In [3]:
# =============================================================================
# All UDFs for chart data prep, TVD, and report building.
# Imported utilities (load_cfg, resolve_params_path) live in utils.py.
# =============================================================================

# ── Chart data helpers ────────────────────────────────────────────────────

def counts_pct(s):
    """Value counts with proportions for a Series.
    Returns DataFrame with columns [category, count, pct]."""
    c = s.value_counts(dropna=False).rename_axis("category").reset_index(name="count")
    c["pct"] = (c["count"] / c["count"].sum()).round(4)
    return c

def quintile_bins(df, col, label, bins=None):
    """Bin a numeric column into quintiles; reuse thresholds when bins is provided.
    Returns (result_df, bins) where result_df has [min, max, count, pct]."""
    s = df[col].dropna()
    if bins is None:
        cut, bins = pd.qcut(s, 5, retbins=True, duplicates="drop")
    else:
        cut = pd.cut(s, bins=bins, include_lowest=True)
    result = cut.value_counts().sort_index().reset_index()
    result.columns = [label, "count"]
    result["pct"] = (result["count"] / result["count"].sum()).round(4)
    result["min"] = [iv.left  for iv in result[label].cat.categories]
    result["max"] = [iv.right for iv in result[label].cat.categories]
    return result[["min", "max", "count", "pct"]], bins

def fy_from_date(s):
    """Return fiscal year string (Jul-start) from a date Series. E.g. FY24, FY25."""
    s = pd.to_datetime(s)
    def _fy(d):
        if pd.isna(d): return None
        return f"FY{(d.year + 1) % 100:02d}" if d.month >= 7 else f"FY{d.year % 100:02d}"
    return s.apply(_fy)

def fy_half(s):
    """Return H1 (Jul–Dec) or H2 (Jan–Jun) from a date Series."""
    return pd.to_datetime(s).dt.month.apply(
        lambda m: "H1 (Jul-Dec)" if m >= 7 else "H2 (Jan-Jun)"
    )

def assign_efs(row):
    """Return the EFS classification string for a project row."""
    rural = row["school_is_underserved_rural"]                    == "Yes"
    race  = row["school_is_historically_underrepresented_race"]   == "Yes"
    inc   = row["school_is_low_income"]                           == "Yes"
    if race and inc: return "Race+Inc"
    if rural:        return "Rural"
    if race:         return "Race"
    if inc:          return "Inc"
    return "NonEFS"

def top5_categories(df_cat, n=10):
    """Top-N item categories by total quantity for a resource_category subset.
    Returns DataFrame with [category, quantity_count, pct]. Default n=10."""
    if df_cat.empty:
        return pd.DataFrame(columns=["category", "quantity_count", "pct"])
    top = (
        df_cat.groupby("item_category", dropna=True)["quantity_count"]
        .sum().nlargest(n).reset_index()
        .rename(columns={"item_category": "category"})
    )
    total = df_cat["quantity_count"].sum()
    top["pct"] = (top["quantity_count"] / total).round(4) if total > 0 else 0.0
    return top[["category", "quantity_count", "pct"]]                .sort_values("quantity_count", ascending=False)                .reset_index(drop=True)

# ── TVD helpers ───────────────────────────────────────────────────────────

def tvd(df_a, df_b, key_col, val_col="pct"):
    """Total variation distance between two categorical distributions."""
    a = df_a[[key_col, val_col]].assign(**{key_col: df_a[key_col].astype(str)})
    b = df_b[[key_col, val_col]].assign(**{key_col: df_b[key_col].astype(str)})
    m = a.merge(b, on=key_col, how="outer", suffixes=("_a","_b")).fillna(0)
    return round((m[f"{val_col}_a"] - m[f"{val_col}_b"]).abs().sum() / 2, 4)

def race_dist(df_a, df_b):
    """Sum of absolute race-distribution differences, normalised to 0–1."""
    m = df_a.merge(df_b, on="race", suffixes=("_a","_b"))
    return round((m["weighted_avg_pct_a"] - m["weighted_avg_pct_b"]).abs().sum() / 100, 4)

# ── Report-builder helpers ────────────────────────────────────────────────

# Fields included in the detail-panel charts (order matters for display)
DETAIL_FIELDS = ["metro", "grade", "efs", "race", "item_category", "state"]

EFS_ORDER  = ["Race+Inc", "Rural", "Race", "Inc", "NonEFS"]

RACE_COLS  = {
    "Black":  "school_percent_black_imputed",
    "Latinx": "school_percent_latinx_imputed",
    "Asian":  "school_percent_asian_imputed",
    "White":  "school_percent_white_imputed",
}

FIELD_META = {
    "metro":         {"val_col": "pct",              "sort_col": "category"},
    "grade":         {"val_col": "pct",              "sort_col": "category"},
    "efs":           {"val_col": "pct",              "sort_col": "category"},
    "race":          {"val_col": "weighted_avg_pct", "sort_col": "race"},
    "item_category": {"val_col": "pct",   "sort_col": "quantity_count", "label_col": "category"},
    "state":         {"val_col": "pct",              "sort_col": "category"},
}

SEP_SUFFIX = {
    "metro": "school coverage",
    "grade": "coverage",
    "efs":   "school coverage",
    "race":  "student population",
}

def _safe_float(v):
    """Convert v to float; return 0.0 for NaN, Inf, or unconvertible values."""
    try:
        f = float(v)
        return 0.0 if math.isnan(f) or math.isinf(f) else f
    except (TypeError, ValueError):
        return 0.0

def df_to_chart(df, field):
    """Convert a chart DataFrame to (labels, values) lists using FIELD_META config."""
    meta      = FIELD_META[field]
    val_col   = meta["val_col"]
    sort_col  = meta["sort_col"]
    label_col = meta.get("label_col", sort_col)
    df_s = df.copy()
    if field == "efs":
        order_map = {v: i for i, v in enumerate(EFS_ORDER)}
        df_s["_ord"] = df_s["category"].map(order_map).fillna(99)
        df_s = df_s.sort_values("_ord").reset_index(drop=True)
    else:
        df_s = df_s.sort_values(sort_col).reset_index(drop=True)
    labels = df_s[label_col].astype(str).tolist()
    values = [_safe_float(v) for v in df_s[val_col]]
    return labels, values

def compute_separators(ins_charts, fc_charts_local):
    """Find top-2 field+value combos where this insight is most above-average vs corpus.
    Race values are on 0–100 scale; others on 0–1. Both converted to ppt for comparison."""
    candidates = []
    for field in DETAIL_FIELDS:
        ch = ins_charts.get(field)
        fc = fc_charts_local.get(field)
        if not ch or not fc:
            continue
        is_race = field == "race"
        fc_map  = dict(zip(fc["labels"], fc["values"]))
        for label, val in zip(ch["labels"], ch["values"]):
            diff_ppt = (val - fc_map.get(label, 0)) * (1 if is_race else 100)
            if diff_ppt > 0.5:
                candidates.append({
                    "field":    field,
                    "label":    label,
                    "diff_ppt": diff_ppt,
                    "suffix":   SEP_SUFFIX.get(field, ""),
                })
    candidates.sort(key=lambda x: x["diff_ppt"], reverse=True)
    result = {}
    for n, key in enumerate(["sep1", "sep2"], 1):
        if len(candidates) >= n:
            c = candidates[n - 1]
            result[f"{key}_val"] = round(c["diff_ppt"], 0)
            result[f"{key}_lbl"] = f"More {c['label']} {c['suffix']}".strip()
        else:
            result[f"{key}_val"] = None
            result[f"{key}_lbl"] = None
    return result

def _extract_body(d):
    """Extract (what_we_see, why_easy_to_miss) text from a structured insight dict."""
    what = (d.get("what_seeing") or d.get("what_we_see") or d.get("what")
            or d.get("finding")  or d.get("body")        or d.get("description") or "")
    why  = (d.get("why_easy_to_miss") or d.get("why")    or d.get("implication")
            or d.get("body2")          or d.get("context") or "")
    return str(what).strip(), str(why).strip()

def _walk(obj, out):
    """Recursively walk structured JSON and populate out[insight_id] with body text."""
    if isinstance(obj, dict):
        if "insight_id" in obj and obj["insight_id"] not in out:
            what, why = _extract_body(obj)
            out[obj["insight_id"]] = {"what_we_see": what, "why_easy_to_miss": why}
        for v in obj.values():
            _walk(v, out)
    elif isinstance(obj, list):
        for item in obj:
            _walk(item, out)

def _is_cross(row):
    """Return True if this curated_df row represents a cross-category insight."""
    rs = str(row.get("report_section", "")).lower()
    s  = str(row.get("section",        "")).lower()
    return ("cross" in rs or s == "key_insights"
            or not row.get("category_bucket")
            or str(row.get("category_bucket", "")).lower() in ("none", ""))

def _fmt_filter(f):
    """Format a single CFG filter dict as a human-readable string."""
    field = f.get("field", "?")
    op    = f.get("op", "?")
    if op == "eq":
        return f"{field} = {f.get('value', '?')}"
    if op == "in":
        vals = f.get("values", f.get("value", "?"))
        if isinstance(vals, list):
            extra = f" (+{len(vals)-5} more)" if len(vals) > 5 else ""
            vals  = ", ".join(str(v) for v in vals[:5]) + extra
        return f"{field} in [{vals}]"
    if op == "range":
        lo = f.get("min", "?")
        hi = f.get("max") or "present"
        return f"{field} from {lo} to {hi}"
    if op == "not_null": return f"{field} is present"
    if op == "is_null":  return f"{field} is absent"
    return f"{field} {op}"

## 3 · Chart Data

In [4]:
# ── Derived columns ───────────────────────────────────────────────────────
df_full = df.copy()
df_full["efs_category"] = df_full.apply(assign_efs, axis=1)
df_full["_fy"]          = fy_from_date(df_full["posted_date"])
df_full["_half"]        = fy_half(df_full["posted_date"])
df_full["_exp_years"]   = (pd.to_datetime(df_full["posted_date"]).dt.year
                           - df_full["teacher_start_teaching_year"])

# ── Enrich insight-project bridge ─────────────────────────────────────────
df_enriched = insight_project_df.merge(df, on="project_id", how="left")
df_enriched["efs_category"] = df_enriched.apply(assign_efs, axis=1)
df_enriched["_fy"]          = fy_from_date(df_enriched["posted_date"])
df_enriched["_half"]        = fy_half(df_enriched["posted_date"])
df_enriched["_exp_years"]   = (pd.to_datetime(df_enriched["posted_date"]).dt.year
                               - df_enriched["teacher_start_teaching_year"])

# ── Resource joins ────────────────────────────────────────────────────────
cat_enriched  = insight_project_df.merge(resource_category_df, on="project_id", how="inner")
item_enriched = insight_project_df.merge(resource_item_df,     on="project_id", how="inner")

# ── Top-5 resource category table ────────────────────────────────────────
cat_rows = []
for iid, grp in cat_enriched.groupby("insight_id"):
    top5 = top5_categories(grp, n=10)
    top5.insert(0, "rank",       list(range(1, len(top5) + 1)))
    top5.insert(0, "insight_id", iid)
    cat_rows.append(top5)
chart_ready_category_df = (
    pd.concat(cat_rows, ignore_index=True)
    [["insight_id","rank","category","quantity_count","pct"]]
)
chart_ready_category_df.to_csv(OUT("chart_data", "chart_ready_category.csv"), index=False)

# ── Top-10 item names table ───────────────────────────────────────────────
item_rows = []
for iid, grp in item_enriched.groupby("insight_id"):
    top10 = (
        grp.groupby("item_name", dropna=True)["quantity_count"]
        .sum().nlargest(10).reset_index()
    )
    top10.insert(0, "rank",       range(1, len(top10) + 1))
    top10.insert(0, "insight_id", iid)
    item_rows.append(top10)
chart_ready_items_df = (
    pd.concat(item_rows, ignore_index=True)
    [["insight_id","rank","item_name","quantity_count"]]
)
chart_ready_items_df.to_csv(OUT("chart_data", "chart_ready_items.csv"), index=False)

# ── Build insight_charts dict ─────────────────────────────────────────────
# Keys: 'full_corpus', 'total_sample', <insight_id>, ...
# Each value: dict of field → DataFrame

_cat_lookup = {iid: grp for iid, grp in cat_enriched.groupby("insight_id")}
insight_charts = {}
cost_bins = None
exp_bins  = None

CHART_SOURCES = (
    [("full_corpus", df_full), ("total_sample", df_enriched)]
    + list(df_enriched.groupby("insight_id"))
)

for insight_id, df_i in CHART_SOURCES:
    enroll = df_i["school_enrollment"]

    # Cost & experience: bin thresholds locked to full_corpus on first pass
    cost_result, cost_bins = quintile_bins(
        df_i.dropna(subset=["total_cost"]), "total_cost", "cost_bucket", bins=cost_bins
    )
    exper_result, exp_bins = quintile_bins(
        df_i.dropna(subset=["_exp_years"]), "_exp_years", "experience_bucket", bins=exp_bins
    )

    # Posting period with chronological sort key
    posting = (
        df_i.assign(
            half_short=df_i["_half"].str.extract(r"^(H[12])", expand=False),
            fy_num=pd.to_numeric(
                df_i["_fy"].str.replace("FY", "", regex=False), errors="coerce"
            ),
        )
        .groupby(["_fy","half_short","fy_num"], dropna=False)
        .size().reset_index(name="count")
        .rename(columns={"_fy":"fy"})
    )
    posting["category"]    = posting["fy"] + " " + posting["half_short"]
    posting["period_sort"] = (posting["fy_num"] * 10
                              + posting["half_short"].map({"H1":1,"H2":2}).fillna(0))
    posting = posting[posting["fy_num"].notna()].copy()
    posting["pct"] = (posting["count"] / posting["count"].sum()).round(4)

    # State distribution (2-letter abbreviations)
    state_dist = counts_pct(df_i["state"]) if "state" in df_i.columns         else pd.DataFrame(columns=["category","count","pct"])

    # Resource categories for full_corpus and total_sample
    if insight_id == "full_corpus":
        cat_df = resource_category_df
    elif insight_id == "total_sample":
        cat_df = cat_enriched
    else:
        cat_df = _cat_lookup.get(insight_id, pd.DataFrame(columns=["item_category","quantity_count"]))

    insight_charts[insight_id] = {
        "project_count": len(df_i),
        "cost_distr":    cost_result,
        "posting":       posting,
        "exper_distr":   exper_result,
        "metro":         counts_pct(df_i["metro_type_at_time_of_posting"]),
        "grade":         counts_pct(df_i["grade_band"]),
        "efs":           counts_pct(df_i["efs_category"]),
        "funding":       counts_pct(pd.Series(
            np.select(
                [df_i["funded_date"].notna(),
                 pd.to_datetime(df_i.get("expiration_date", pd.Series(dtype=str)),
                                errors="coerce") <= pd.Timestamp.today()],
                ["Funded", "Expired"],
                default="Live"
            ), index=df_i.index
        )),
        "race": pd.DataFrame([
            {
                "race": r,
                "weighted_avg_pct": round(
                    (df_i[col] * enroll).sum() / enroll.sum(), 4
                ) if enroll.sum() > 0 else 0.0
            }
            for r, col in RACE_COLS.items()
            if col in df_i.columns
        ]),
        "item_category": top5_categories(cat_df),
        "state":         state_dist,
    }

print(f"Built chart data for {len(insight_charts)} entries "
      f"(incl. full_corpus + total_sample)")

Built chart data for 49 entries (incl. full_corpus + total_sample)


## 4 · Distinctness (TVD)

In [5]:
# Measures how distinct each insight's distributions are vs the full corpus.
# Only insights with > 500 supporting projects are included.

corpus  = insight_charts["full_corpus"]
total_n = insight_charts["total_sample"]["project_count"]
tvd_cols = ["funding", "cost_distr", "exper_distr", "metro", "grade", "efs", "race"]

TVD_KEY = {
    "funding": "category", "cost_distr": "min", "exper_distr": "min",
    "metro": "category",   "grade": "category", "efs": "category",
    "race": "race",
}

distinctness = []
for insight_id, charts in insight_charts.items():
    if insight_id in ("full_corpus", "total_sample"):
        continue
    row = {
        "insight_id":    insight_id,
        "project_count": charts["project_count"],
        "pct_of_total":  round(charts["project_count"] / total_n, 4),
    }
    for field in tvd_cols:
        key = TVD_KEY[field]
        if field == "race":
            row[field] = race_dist(charts["race"], corpus["race"])
        else:
            row[field] = tvd(charts[field], corpus[field], key)
    distinctness.append(row)

distinctness_df = (
    pd.DataFrame(distinctness)
    .pipe(lambda d: d[d["project_count"] > 500])
    .reset_index(drop=True)
)
distinctness_df["mean_tvd"] = (
    distinctness_df[tvd_cols].mean(axis=1).round(4)
)
distinctness_df = (
    distinctness_df.sort_values("mean_tvd", ascending=False)
    .reset_index(drop=True)
)

print(f"Distinctness table: {len(distinctness_df)} insights with > 500 projects")
distinctness_df.head(10)

Distinctness table: 42 insights with > 500 projects


,insight_id,project_count,pct_of_total,funding,cost_distr,exper_distr,metro,grade,efs,race,mean_tvd
0,BG_034,1871,0.0117,0.3326,0.1013,0.0551,0.1612,0.5239,0.1559,0.2343,0.2235
1,BG_012,1530,0.0096,0.3326,0.3098,0.0671,0.1023,0.4157,0.0604,0.0834,0.1959
2,KI_002,9014,0.0565,0.3326,0.1894,0.0260,0.1399,0.2228,0.1657,0.1863,0.1804
3,BG_029,750,0.0047,0.3326,0.1653,0.0322,0.0992,0.4408,0.0646,0.0458,0.1686
4,BG_020,6891,0.0432,0.3326,0.1561,0.0420,0.1182,0.2554,0.1293,0.1376,0.1673
5,BG_021,8154,0.0511,0.3326,0.1826,0.0264,0.1014,0.2443,0.1416,0.1214,0.1643
6,BG_007,6967,0.0437,0.3326,0.1609,0.0522,0.1199,0.1917,0.1224,0.1224,0.1574
7,BG_028,588,0.0037,0.3326,0.1254,0.0801,0.0911,0.3400,0.0656,0.0597,0.1564
8,BG_022,7659,0.0480,0.3326,0.2655,0.0509,0.1108,0.0653,0.1150,0.1194,0.1514
9,BG_006,3714,0.0233,0.3326,0.1084,0.0558,0.1573,0.1070,0.1158,0.1530,0.1471


## 5 · Map Export

In [6]:
# Per-insight project count keyed by 2-letter state abbreviation.
# The 'full_corpus' row gives the baseline distribution.

map_rows = []

for insight_id, df_i in [("full_corpus", df_full)] + list(df_enriched.groupby("insight_id")):
    if "state" not in df_i.columns:
        continue
    counts = (
        df_i["state"]
        .value_counts(dropna=True)
        .rename_axis("state")
        .reset_index(name="project_count")
    )
    total = counts["project_count"].sum()
    counts["pct"] = (counts["project_count"] / total).round(4)
    counts.insert(0, "insight_id", insight_id)
    map_rows.append(counts)

if map_rows:
    map_df = pd.concat(map_rows, ignore_index=True)
    map_df.to_csv(OUT("chart_data", "chart_ready_map.csv"), index=False)
    print(f"Map table: {len(map_df):,} rows | "
          f"{map_df['state'].nunique()} states | "
          f"{map_df['insight_id'].nunique()} insights (incl. full_corpus)")
    display(map_df[map_df["insight_id"] == "full_corpus"].head(10))
else:
    print("WARNING: 'state' column not found — check project_attributes.csv join in Section 1.")

Map table: 2,234 rows | 51 states | 48 insights (incl. full_corpus)


,insight_id,state,project_count,pct
0,full_corpus,CA,305472,0.1355
1,full_corpus,NY,182543,0.0810
2,full_corpus,TX,176646,0.0783
3,full_corpus,FL,119496,0.0530
4,full_corpus,IL,86225,0.0382
5,full_corpus,NC,80293,0.0356
6,full_corpus,AZ,73493,0.0326
7,full_corpus,OK,69494,0.0308
8,full_corpus,PA,64781,0.0287
9,full_corpus,NV,61068,0.0271


## 6 · Report Builder

In [8]:
# Reads outputs from Sections 3–5 and writes a self-contained HTML report.

# ── Emoji favicon ─────────────────────────────────────────────────────────
_emoji = "✏️"   # 📊 🎓 📚 ✏️
_svg   = f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 100 100"><text y=".9em" font-size="90">{_emoji}</text></svg>'
favicon_uri = "data:image/svg+xml;base64," + base64.b64encode(_svg.encode()).decode()

# ── Full corpus charts + global per-field xmax ────────────────────────────
fc_charts_local = {}
field_xmax      = {}

for field in DETAIL_FIELDS:
    fc_df = insight_charts.get("full_corpus", {}).get(field)
    if fc_df is None or fc_df.empty:
        continue
    labels, values = df_to_chart(fc_df, field)
    fc_charts_local[field] = {"labels": labels, "values": values}
    field_xmax[field] = max(values) * 1.2 if values else 1.0

# ── Comprehensive corpus category baseline ───────────────────────────────────
# Uses all categories (not just top 10) so niche categories like Candy, Chips,
# Party Favors still get a baseline tick showing their share in the full corpus.
if not resource_category_df.empty:
    _corp_total = resource_category_df["quantity_count"].sum()
    _corpus_cat_map = (
        resource_category_df
        .groupby("item_category", dropna=True)["quantity_count"]
        .sum()
        .div(_corp_total)
        .round(6)
        .to_dict()
    )
else:
    _corpus_cat_map = {}

for iid, charts in insight_charts.items():
    if iid in ("full_corpus", "total_sample"):
        continue
    for field in DETAIL_FIELDS:
        ch = charts.get(field)
        if ch is None or (hasattr(ch, "empty") and ch.empty):
            continue
        _, values = df_to_chart(ch, field)
        candidate = max(values) * 1.2 if values else 0.0
        if candidate > field_xmax.get(field, 0):
            field_xmax[field] = candidate

for field in field_xmax:
    field_xmax[field] = round(field_xmax[field], 4)
    fc_charts_local.setdefault(field, {})["xmax"] = field_xmax[field]

# ── Extract prose body text ───────────────────────────────────────────────
insight_bodies = {}
_walk(structured, insight_bodies)

# ── Item names: insight_id → [[name, qty], ...] ───────────────────────────
# Stored as pairs so the template can render both name and quantity.
item_names_lookup = (
    chart_ready_items_df
    .sort_values("rank")
    .groupby("insight_id")
    .apply(lambda g: list(zip(g["item_name"], g["quantity_count"].astype(int))))
    .to_dict()
)

# ── TVD lookup ────────────────────────────────────────────────────────────
tvd_lookup = {row["insight_id"]: row.to_dict() for _, row in distinctness_df.iterrows()}

# ── Filter description from CFG ───────────────────────────────────────────
try:
    _filters_list  = CFG.get("analysis", {}).get("filters", [])
    _filter_logic  = CFG.get("analysis", {}).get("filter_logic", "and").upper()
    _groupby_label = GROUPBY_FIELD.replace("_", " ")
    if _filters_list:
        _filter_str     = f" {_filter_logic} ".join(_fmt_filter(f) for f in _filters_list)
        filter_headline = (f"Insights were found by deep diving on {_groupby_label}, "
                           f"filtered on {_filter_str}")
        filter_detail   = (f"{len(_filters_list)} filter{'s' if len(_filters_list) != 1 else ''} "
                           f"applied ({_filter_logic}) · {len(curated_df)} insights accepted "
                           f"from {len(df):,} projects")
    else:
        filter_headline = (f"Insights were found by deep diving on {_groupby_label} "
                           f"across {len(df):,} projects")
        filter_detail   = f"No filters applied · {len(curated_df)} insights accepted"
except Exception as e:
    filter_headline = f"Insights were found by deep diving on {GROUPBY_FIELD.replace('_',' ')}"
    filter_detail   = f"No filters applied · {len(curated_df)} insights accepted"
    print(f"Warning: could not read filter config ({e})")

# ── Build insights list ───────────────────────────────────────────────────
insights_out = []

for _, row in curated_df.sort_values("supporting_project_count", ascending=False).iterrows():
    iid      = row["insight_id"]
    body     = insight_bodies.get(iid, {"what_we_see": "", "why_easy_to_miss": ""})
    tvd_r    = tvd_lookup.get(iid, {})
    is_cross = _is_cross(row)

    # Build per-field chart dicts with labels, values, baseline, xmax
    charts_out = {}
    for field in DETAIL_FIELDS:
        ic = insight_charts.get(iid, {}).get(field)
        if ic is None or (hasattr(ic, "empty") and ic.empty):
            continue
        labels, values = df_to_chart(ic, field)
        if not labels:
            continue
        fc_lbls = fc_charts_local.get(field, {}).get("labels", labels)
        fc_vals = fc_charts_local.get(field, {}).get("values", [0.0] * len(labels))
        fc_map  = dict(zip(fc_lbls, fc_vals))
        charts_out[field] = {
            "labels":   labels,
            "values":   values,
            "baseline": [_corpus_cat_map.get(lbl, 0.0) if field == "item_category" else fc_map.get(lbl) for lbl in labels],
            "xmax":     field_xmax.get(field, 1.0),
        }

    seps = compute_separators(charts_out, fc_charts_local)

    insights_out.append({
        "id":                   iid,
        "title":                str(row.get("title", row.get("theme", iid))),
        "what_we_see":          body["what_we_see"],
        "why_easy_to_miss":     body["why_easy_to_miss"],
        "category_bucket":      str(row.get("category_bucket", "")),
        "is_cross":             is_cross,
        "section":              str(row.get("section", "")),
        "report_section":       str(row.get("report_section", "")),
        "project_count":        int(_safe_float(row.get("supporting_project_count", 0))),
        "pct_of_total":         round(
            _safe_float(row.get("supporting_project_count", 0)) / max(ANALYSIS_N, 1), 4
        ),
        "verified_topic_count": int(_safe_float(row.get("verified_topic_count", 0))),
        "mean_topic_share":     round(_safe_float(
            row.get("mean_topic_share_all_verified_topics", 0)), 3),
        "verification_ratio":   round(_safe_float(row.get("verification_ratio", 0)), 3),
        "looker_url":           str(row.get("looker_url", "")),
        "sep1_val":             seps["sep1_val"],
        "sep1_lbl":             seps["sep1_lbl"],
        "sep2_val":             seps["sep2_val"],
        "sep2_lbl":             seps["sep2_lbl"],
        "item_names":           item_names_lookup.get(iid, []),
        "tvd": {
            "metro": round(_safe_float(tvd_r.get("metro", 0)), 4),
            "grade": round(_safe_float(tvd_r.get("grade", 0)), 4),
            "efs":   round(_safe_float(tvd_r.get("efs",   0)), 4),
            "race":  round(_safe_float(tvd_r.get("race",  0)), 4),
            "mean":  round(_safe_float(tvd_r.get("mean_tvd", 0)), 4),
        },
        "charts": charts_out,
    })

# Cross-category first, then main-section, then by project count descending
insights_out.sort(key=lambda x: (
    0 if x["is_cross"] else 1,
    0 if "main" in x["report_section"].lower() else 1,
    -x["project_count"],
))

# ── Account metadata ──────────────────────────────────────────────────────
account_meta_path = ROOT / "account_meta.json"
account_meta = (
    json.loads(account_meta_path.read_text())
    if account_meta_path.exists()
    else {"name": "", "logo": "", "date": RUN_DATE}
)

# ── Assemble JSON payload ─────────────────────────────────────────────────
payload = {
    "meta": {
        "run_id":          RUN_ID,
        "run_date":        RUN_DATE,
        "project_count":   ANALYSIS_N,   # filtered scope from filter_summary.json
        "insight_count":   len(curated_df),
        "groupby_field":   GROUPBY_FIELD,
        "filter_headline": filter_headline,
        "filter_detail":   filter_detail,
    },
    "account":     account_meta,
    "full_corpus": fc_charts_local,
    "insights":    insights_out,
}

# ── Read template + Chart.js, inject payload, write HTML ──────────────────
template_path = ROOT / "report_template.html"
chartjs_path  = ROOT / "chart.umd.min.js"

if not template_path.exists():
    raise FileNotFoundError(f"report_template.html not found at {template_path}")
if not chartjs_path.exists():
    raise FileNotFoundError(
        f"chart.umd.min.js not found at {chartjs_path}\n"
        "Download from cdn.jsdelivr.net/npm/chart.js/dist/chart.umd.min.js"
    )

template = template_path.read_text(encoding="utf-8")
chartjs  = chartjs_path.read_text(encoding="utf-8")

html_out = (
    template
    .replace("__CHARTJS__",     chartjs)
    .replace("__FAVICON__",     favicon_uri)
    .replace("__REPORT_DATA__", json.dumps(payload, ensure_ascii=False, separators=(",",":")))
)

html_path = OUT("reports", "trend_tracker_report.html")
html_path.write_text(html_out, encoding="utf-8")

# ── Summary ───────────────────────────────────────────────────────────────
size_kb  = len(html_out.encode("utf-8")) / 1024
body_ct  = sum(1 for i in insights_out if i["what_we_see"])
sep_ct   = sum(1 for i in insights_out if i["sep1_val"] is not None)
cross_ct = sum(1 for i in insights_out if i["is_cross"])

print(f"HTML report → {html_path}")
print(f"Size: {size_kb:.0f} KB  ·  Insights: {len(insights_out)}  ·  Projects: {len(df):,}")
print(f"Body text: {body_ct}/{len(insights_out)}  ·  "
      f"Separators: {sep_ct}/{len(insights_out)}  ·  "
      f"Cross-category: {cross_ct}")
print(f"Filter: {filter_headline[:120]}")

HTML report → OUTPUTS/runs/project_category/2026-04-11/20260411_201415_project_category_77e319db/reports/trend_tracker_report.html
Size: 518 KB  ·  Insights: 47  ·  Projects: 2,255,909
Body text: 47/47  ·  Separators: 47/47  ·  Cross-category: 8
Filter: Insights were found by deep diving on project category, filtered on funded_date from 2023-07-01 to present


/var/folders/j3/jwjf6cwj7czdz1klxbhhjst80000gp/T/ipykernel_12879/1709439278.py:62: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g["item_name"], g["quantity_count"].astype(int))))
